# 물류공간 예측 데이터셋 → YOLOv8n 학습 → Unity Sentis ONNX 변환

**AI Hub 데이터셋**: 물류공간 예측 데이터 (dataSetSn=71861)  
**사전 준비**: aihub.or.kr 로그인 → 마이페이지 → API 인증키 발급  
**런타임**: GPU (T4 이상 권장, Colab 무료 사용 가능)

## 0. 환경 설치

In [ ]:
!pip install ultralytics requests tqdm -q
import os, json, shutil, zipfile, requests
from pathlib import Path
from tqdm import tqdm
print('✅ 설치 완료')

## 1. AI Hub API 인증키 설정

In [ ]:
# ─── 여기에 aihub.or.kr 마이페이지에서 발급받은 API 키를 입력하세요 ───
AIHUB_API_KEY = 'YOUR_API_KEY_HERE'
DATASET_SN    = '71861'   # 물류공간 예측 데이터

BASE_DIR  = Path('/content/logistics')
BASE_DIR.mkdir(exist_ok=True)

HEADERS = {'apikey': AIHUB_API_KEY}
print(f'데이터셋 SN: {DATASET_SN}')

## 2. 파일 목록 조회 및 다운로드

In [ ]:
def get_file_list(dataset_sn: str) -> list:
    """AI Hub API로 데이터셋 파일 목록을 조회합니다."""
    url = 'https://api.aihub.or.kr/api/aihubdata/getDatasetFileList.do'
    params = {'dataSetSn': dataset_sn}
    resp = requests.get(url, headers=HEADERS, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    files = data.get('result', {}).get('fileList', [])
    print(f'총 {len(files)}개 파일 목록 수신')
    for f in files:
        print(f"  [{f.get('fileSn')}] {f.get('fileNm')}  {f.get('fileSize',0)/1e9:.2f} GB")
    return files

file_list = get_file_list(DATASET_SN)

In [ ]:
def download_file(dataset_sn: str, file_sn: str, save_path: Path) -> Path:
    """AI Hub API로 특정 파일을 다운로드합니다."""
    url = f'https://api.aihub.or.kr/down/0.5/{dataset_sn}.do'
    params = {'fileSn': file_sn}

    with requests.get(url, headers=HEADERS, params=params,
                      stream=True, timeout=300) as r:
        r.raise_for_status()
        total = int(r.headers.get('Content-Length', 0))
        with open(save_path, 'wb') as f, tqdm(
            total=total, unit='B', unit_scale=True,
            desc=save_path.name
        ) as bar:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                bar.update(len(chunk))
    return save_path

# 이미지 + 라벨 파일만 선택적으로 다운로드
# file_sn 값은 위 get_file_list 결과에서 확인
# 학습용(Training) 이미지와 라벨을 우선 다운로드
TARGET_FILE_SNS = [
    # file_list 결과를 보고 여기에 원하는 fileSn 을 추가하세요
    # 예: '1', '2', '3'
]

zip_dir = BASE_DIR / 'zips'
zip_dir.mkdir(exist_ok=True)

downloaded = []
for file_info in file_list:
    sn = str(file_info.get('fileSn', ''))
    if TARGET_FILE_SNS and sn not in TARGET_FILE_SNS:
        continue
    name = file_info.get('fileNm', f'file_{sn}.zip')
    dest = zip_dir / name
    if dest.exists():
        print(f'⏭️  이미 존재: {name}')
    else:
        download_file(DATASET_SN, sn, dest)
    downloaded.append(dest)

print(f'\n다운로드 완료: {len(downloaded)}개 파일')

## 3. 압축 해제

In [ ]:
extract_dir = BASE_DIR / 'raw'
extract_dir.mkdir(exist_ok=True)

for zf in downloaded:
    print(f'압축 해제: {zf.name}')
    with zipfile.ZipFile(zf, 'r') as z:
        z.extractall(extract_dir)

# 디렉토리 구조 확인
!find /content/logistics/raw -maxdepth 3 -type d | head -30

## 4. JSON 어노테이션 → YOLO 형식 변환

In [ ]:
# ─── 클래스 정의 (AI Hub 11개 대분류 기준) ───
# 실제 JSON 파일의 category 이름에 맞게 수정하세요
CLASSES = [
    '가공식품',      # 0
    '신선식품',      # 1
    '생활용품',      # 2
    '패션의류',      # 3
    '스포츠',        # 4
    '레저용품',      # 5
    '디지털_가전',   # 6
    '가구',          # 7
    '패션잡화',      # 8
    '도서_완구',     # 9
    '기타',          # 10
]
CLASS2ID = {c: i for i, c in enumerate(CLASSES)}
print('클래스 목록:', CLASS2ID)

In [ ]:
def coco_bbox_to_yolo(bbox, img_w, img_h):
    """COCO [x_min, y_min, w, h] → YOLO [cx, cy, w, h] 정규화"""
    x, y, w, h = bbox
    cx = (x + w / 2) / img_w
    cy = (y + h / 2) / img_h
    nw = w / img_w
    nh = h / img_h
    return cx, cy, nw, nh

def convert_json_to_yolo(json_path: Path, img_dir: Path,
                          out_img_dir: Path, out_lbl_dir: Path,
                          class2id: dict) -> int:
    """
    AI Hub JSON 어노테이션 → YOLO txt 변환.
    COCO 형식 (images / annotations / categories) 기준.
    """
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    with open(json_path, encoding='utf-8') as f:
        data = json.load(f)

    # 이미지 ID → 메타 매핑
    id2img = {img['id']: img for img in data.get('images', [])}

    # 카테고리 매핑 (AI Hub category id → 우리 class id)
    cat_remap = {}
    for cat in data.get('categories', []):
        name = cat.get('name', '')
        # 대분류 이름으로 매핑 (supercategory 또는 name 사용)
        key = cat.get('supercategory', name)
        if key in class2id:
            cat_remap[cat['id']] = class2id[key]
        elif name in class2id:
            cat_remap[cat['id']] = class2id[name]

    # 이미지별 어노테이션 그룹화
    from collections import defaultdict
    ann_map = defaultdict(list)
    for ann in data.get('annotations', []):
        ann_map[ann['image_id']].append(ann)

    converted = 0
    for img_id, img_info in id2img.items():
        file_name = img_info['file_name']
        img_w = img_info['width']
        img_h = img_info['height']

        src_img = img_dir / file_name
        if not src_img.exists():
            continue

        anns = ann_map.get(img_id, [])
        lines = []
        for ann in anns:
            cat_id = ann.get('category_id')
            cls = cat_remap.get(cat_id)
            if cls is None:
                continue
            bbox = ann.get('bbox', [])
            if len(bbox) < 4:
                continue
            cx, cy, w, h = coco_bbox_to_yolo(bbox, img_w, img_h)
            # 유효 범위 클램프
            cx, cy, w, h = (max(0, min(1, v)) for v in (cx, cy, w, h))
            if w <= 0 or h <= 0:
                continue
            lines.append(f'{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        if not lines:
            continue

        stem = Path(file_name).stem
        shutil.copy(src_img, out_img_dir / src_img.name)
        (out_lbl_dir / f'{stem}.txt').write_text('\n'.join(lines))
        converted += 1

    return converted

print('변환 함수 준비 완료')

In [ ]:
# ─── 경로 설정: 실제 해제된 폴더 구조에 맞게 수정하세요 ───
RAW = extract_dir

# 예시 경로 (AI Hub 구조에 따라 달라질 수 있음)
TRAIN_JSON = RAW / 'Training' / 'Labeling' / 'train.json'
TRAIN_IMG  = RAW / 'Training' / 'Image'
VAL_JSON   = RAW / 'Validation' / 'Labeling' / 'val.json'
VAL_IMG    = RAW / 'Validation' / 'Image'

YOLO_DIR = BASE_DIR / 'yolo_dataset'

print('Train JSON 변환 중...')
n_train = convert_json_to_yolo(
    TRAIN_JSON, TRAIN_IMG,
    YOLO_DIR / 'images' / 'train',
    YOLO_DIR / 'labels' / 'train',
    CLASS2ID
)
print(f'  → {n_train}장 변환')

print('Validation JSON 변환 중...')
n_val = convert_json_to_yolo(
    VAL_JSON, VAL_IMG,
    YOLO_DIR / 'images' / 'val',
    YOLO_DIR / 'labels' / 'val',
    CLASS2ID
)
print(f'  → {n_val}장 변환')
print(f'\n총 {n_train + n_val}장 변환 완료')

## 5. data.yaml 생성

In [ ]:
yaml_content = f"""path: {YOLO_DIR}
train: images/train
val:   images/val

nc: {len(CLASSES)}
names: {CLASSES}
"""

yaml_path = YOLO_DIR / 'data.yaml'
yaml_path.write_text(yaml_content)
print(yaml_content)

## 6. YOLOv8n 학습

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # Nano 모델 (모바일 최적)

results = model.train(
    data    = str(yaml_path),
    epochs  = 100,
    imgsz   = 640,
    batch   = 16,       # VRAM 부족 시 8로 줄이세요
    device  = 0,        # GPU 사용 (CPU는 'cpu')
    name    = 'logistics_yolov8n',
    project = str(BASE_DIR / 'runs'),
    # 모바일 성능 최적화 옵션
    patience   = 20,    # Early stopping (20 epoch 개선 없으면 중단)
    optimizer  = 'AdamW',
    lr0        = 0.001,
    augment    = True,
    mixup      = 0.1,
    copy_paste = 0.1,
)

best_pt = BASE_DIR / 'runs' / 'logistics_yolov8n' / 'weights' / 'best.pt'
print(f'\n✅ 학습 완료: {best_pt}')

## 7. ONNX 변환 (Unity Sentis 호환)

In [ ]:
best_model = YOLO(str(best_pt))

# Unity Sentis 호환 설정
#   opset=12  : Sentis 2.x 권장
#   simplify  : 불필요한 노드 제거 → 모바일 최적화
#   dynamic   : 배치 크기를 동적으로 (1로 고정 시 False)
export_path = best_model.export(
    format   = 'onnx',
    opset    = 12,
    simplify = True,
    dynamic  = False,   # Unity Sentis 는 고정 배치 권장
    imgsz    = 640,
)

print(f'\n✅ ONNX 변환 완료: {export_path}')
print('이 파일을 Unity Assets/Models/ 폴더에 복사하세요.')

## 8. 검증 및 클래스 이름 확인

In [ ]:
import onnx
!pip install onnx -q

m = onnx.load(export_path)
onnx.checker.check_model(m)

inp = m.graph.input[0]
out = m.graph.output[0]
print('✅ ONNX 모델 검증 통과')
print(f'입력 shape : {[d.dim_value for d in inp.type.tensor_type.shape.dim]}')
print(f'출력 shape : {[d.dim_value for d in out.type.tensor_type.shape.dim]}')
print(f'\n클래스 목록 (YoloDetector.classNames[] 에 동일하게 입력하세요):')
for i, c in enumerate(CLASSES):
    print(f'  [{i}] {c}')

## 9. Unity 설정 가이드

1. `best.onnx` → Unity `Assets/Models/yolov8n_logistics.onnx` 로 복사
2. Project 창에서 해당 파일 선택 → Inspector에서 **Sentis Model** 자동 인식
3. `YoloDetector` 컴포넌트 → **ModelAsset** 필드에 드래그
4. **Class Names[]** 에 위 클래스 목록 순서대로 입력
5. `ProductDatabase.asset` 에 각 classId 별 물리 특성 입력

### 출력 텐서 shape 확인
```
입력:  [1, 3, 640, 640]
출력:  [1, 4+11, 8400]  = [1, 15, 8400]
         ↑        ↑
       bbox   클래스수  앵커수
```
YoloDetector.cs 의 PostProcess() 는 이 shape 을 자동으로 파싱합니다.